# Библиотеки

In [8]:
import pandas as pd
from pathlib import Path

# Загрузка данных

In [9]:
responses = pd.read_json(
    (
        Path("../outputs/runs/ex1")
        / Path("ex1_20260923T192438Z_39d7348a")
        / Path("responses.jsonl")
    ),
    lines=True
)
failures = pd.read_json(
    (
        Path("../outputs/runs/ex1")
        / Path("ex1_20260923T192438Z_39d7348a")
        / Path("failures.jsonl")
    ),
    lines=True
)

In [10]:
responses.head(1)

,run_id,trial_index,H0,H1,right_hypothesis,s,r,k,X,sse_h0,...,bic_h1,delta_bic,log_likelihood_h0,log_likelihood_h1,log_likelihood_ratio,monte_carlo,prior,llm_output,answer,is_right
0,ex1_20260923T192438Z_39d7348a,1,f(x) = k*x,f(x) = k*x + alpha*x^{3},f(x) = k*x + alpha*x^{3},0.01,0.05,1,1,0.01592,...,-290.989172,64.74778,30.987207,102.376741,71.389535,1.0,neutral,H1,H1,True


In [12]:
responses["uroven"] = pd.cut(
    x=responses["monte_carlo"],
    bins=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    right=True,
    include_lowest=True
)

In [21]:
(
    responses[responses["prior"] == "neutral"]
    .groupby("uroven", observed=True)
    .agg(
        n=("prior", "size"),
        avg_evidence_strength=("monte_carlo", "mean"),
        llm_p_h1=("is_right", "mean")
    )
    .reset_index()
)

,uroven,n,avg_evidence_strength,llm_p_h1
0,"(0.1, 0.2]",12,0.132083,0.583333
1,"(0.2, 0.3]",12,0.243750,0.916667
2,"(0.3, 0.4]",6,0.372167,0.333333
3,"(0.6, 0.7]",18,0.655000,1.000000
4,"(0.8, 0.9]",6,0.876500,1.000000
5,"(0.9, 1.0]",96,0.995188,0.989583


In [22]:
(
    responses[responses["prior"] == "wrong"]
    .groupby("uroven", observed=True)
    .agg(
        n=("prior", "size"),
        avg_evidence_strength=("monte_carlo", "mean"),
        llm_p_h1=("is_right", "mean")
    )
    .reset_index()
)

,uroven,n,avg_evidence_strength,llm_p_h1
0,"(0.1, 0.2]",12,0.132083,0.166667
1,"(0.2, 0.3]",12,0.243750,0.250000
2,"(0.3, 0.4]",6,0.372167,0.166667
3,"(0.6, 0.7]",18,0.655000,0.333333
4,"(0.8, 0.9]",6,0.876500,0.500000
5,"(0.9, 1.0]",96,0.995188,0.760417


Краткий вывод: при neutral prior llm системно чаще приходит к H1, чем это было бы статистически оправдано; при wrong prior llm системно реже приходит к H!, чем это было бы статистически оправдано.